# Figure 4 code, plus Figure 5A and Supplemental Figure 5A and 5B

## 0. Setup

In [ ]:
# [250331 cell 0]
import os
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import smqpp
import re
import anndata
import seaborn as sns

import warnings 
warnings.simplefilter(action='ignore',category=FutureWarning)
warnings.filterwarnings(action='ignore')

In [ ]:
# [250331 cell 1]
sc.settings.verbosity = 3
sc.logging.print_header()

from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list(name="gene_cmap",colors=["lightgrey","thistle","red","darkred"])
sc.settings.set_figure_params(dpi=80,color_map="viridis",vector_friendly=False,dpi_save=300)

In [ ]:
# [250331 cell 2]
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
color_map = LinearSegmentedColormap.from_list('ss2_to_10x', ['lightgrey', 'red', 'darkred'])

In [ ]:
# [250331 cell 3]
import rpy2
%load_ext rpy2.ipython

In [ ]:
# [250331 cell 4]
files_path = "/home/ql321/ql321/files/"
cc_genes_txt = files_path+"regev_lab_cell_cycle_genes_mouse_Corrected.txt"
model_molo_genes_txt = files_path+"model_molo_genes.txt"
p53GL_txt = files_path+"p53GL.txt"
AL_genes_txt = files_path+"age_gene20.txt"
RepopSig_genes_txt = files_path+"RepopSig"

## 1. Upstream objects

### 1.0 The root object
Builds the index sorting table and the corrected Biotin labels.
Writes `df_All_from_cell_sorting_file_250121.csv` and `adata_qc_173_corrected_Biotin_250121.h5ad`.
Nine cells get their Biotin label corrected, and eight of the nine move from HI to LO.

In [ ]:
# [250121 cell 11]
adata_qc = sc.read_h5ad("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/SLX21561_qc.h5ad")
adata_qc

In [ ]:
# [250121 cell 12]
sc.pp.filter_genes(adata_qc,min_counts=1)

In [ ]:
# [250121 cell 21]
metatable_scRNAseq = pd.read_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/scRNAseq_metatable.csv",index_col=0)
metatable_scRNAseq.head()

In [ ]:
# [250121 cell 22]
metatable_scRNAseq["sample_id"] = metatable_scRNAseq.index

In [ ]:
# [250121 cell 23]
def pro_platei(plate_i,sheet_name):#plate_i is Plate 3;sheet_name is EXP51C_P3
    df_i = pd.read_excel(cell_sorting_file,sheet_name=sheet_name)#containing the median mean values of indexes
    meta_i=metatable_scRNAseq[metatable_scRNAseq["Plate_number"]==plate_i]
    df_all = pd.merge(df_i,meta_i,left_on="Well",right_on="Position_in_96_well_plate_sorted",how="inner")
    print(df_all.shape)
    df_all=df_all.copy()
    #df_all["Control_Infected"] = df_all["Condition"].astype(str)
    df_result=df_all.copy()#[["Plate_number","Control_Infected","Unique_sample_descriptor"]+cols]
    return df_result

df_plate1 = pro_platei("Plate 1","EPX51B_P1")
df_plate2 = pro_platei("Plate 2","EXP51B_P2")
df_plate3 = pro_platei("Plate 3","EXP51C_P3")


#generate the median dataframe to be the adata X table
df_All = pd.concat([df_plate1,df_plate2,df_plate3]).reset_index(drop=True)
df_All.head()

In [ ]:
# [250121 cell 30]
df_All.to_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/df_All_from_cell_sorting_file_250121.csv")

In [ ]:
# [250121 cell 114]
def get_df_all_median(df_All):
    cell_sorting_file = "/home/ql321/ql321/work/work_SLX21561_SLX21562/index_sorting_analysis0824/index_information.xlsx"
    adata_umap = sc.read_h5ad("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/adata_umap_173.h5ad")
    df = pd.read_excel(cell_sorting_file,sheet_name="EXP51C_P3")
    index_sort_cols_median = df.columns[7:-2:2]
    index_sort_cols_mean = df.columns[6:-2:2]
    print(len(index_sort_cols_mean),index_sort_cols_mean[:5])
    print(len(index_sort_cols_median),index_sort_cols_median[:5])
    #
    col_merge = ["Plate_number","Position_in_96_well_plate_sorted","Condition"]
    col_un_overlapped = [i for i in adata_umap.obs.columns if i not in df_All.columns]
    #
    df_All_merged = pd.merge(adata_umap.obs[col_un_overlapped+col_merge],df_All,left_on=["Plate_number","Position_in_96_well_plate_sorted","Condition"],
                             right_on=["Plate_number","Well","Condition"],how='inner')
    
    df_all_median = df_All_merged[["Plate_number","Well","Biotin","Condition","sample_id"]+list(index_sort_cols_median)]
    #remove Plate 3 A7 A8 which have unknown indexes
    df_all_median = df_all_median.copy()
    df_all_median['symbol'] = df_all_median["Plate_number"]+"_"+df_all_median["Well"]
    df_all_median_info = df_all_median.copy()
    df_all_median_info = df_all_median_info.rename(columns=lambda x:re.sub('P7 | Median','',x))
    df_all_median = df_all_median[index_sort_cols_median]
    df_all_median = df_all_median.rename(columns=lambda x:re.sub('P7 | Median','',x))
    print(df_all_median.head())
    print(df_all_median.shape)
    return df_all_median,df_all_median_info

df_all_median,df_all_median_info = get_df_all_median(df_All)

In [ ]:
# [250121 cell 115]
df_all_median_info

In [ ]:
# [250121 cell 116]
df_all_median_info.to_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/df_all_median_info_250121.csv")

In [ ]:
# [250121 cell 161]
df_All.index=df_All["sample_id"]

In [ ]:
# [250121 cell 162]
df_All.head()

In [ ]:
# [250121 cell 166]
df_all_median_info.index=df_all_median_info["sample_id"]

In [ ]:
# [250121 cell 167]
adata_qc

In [ ]:
# [250121 cell 168]
adata_qc.obs["Biotin"].value_counts(dropna=False)

In [ ]:
# [250121 cell 170]
df_all_median_info.shape

In [ ]:
# [250121 cell 171]
adata_qc_173 = adata_qc[adata_qc.obs_names.isin(list(df_all_median_info.index))].copy()
adata_qc_173

In [ ]:
# [250121 cell 172]
adata_qc_173.obs = pd.merge(adata_qc_173.obs,df_all_median_info[["Biotin BV421 DAPI-A","EPCR PE PE-A"]],left_index=True,right_index=True)

In [ ]:
# [250121 cell 175]
cells11 = pd.read_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/excluded_11cells_250112.csv",index_col=0)
cells11

In [ ]:
# [250121 cell 177]
cells9 = list(cells11.head(9).index)

In [ ]:
# [250121 cell 185]
for index,row in adata_qc_173.obs.iterrows():
    if index in cells9:
        if index == "SLX-21561.i722_i513":
            adata_qc_173.obs.loc[index,"Biotin"] = "HI"
        else:
            adata_qc_173.obs.loc[index,"Biotin"] = "LO"

In [ ]:
# [250121 cell 186]
adata_qc_173.obs[adata_qc_173.obs_names.isin(cells9)][["Biotin","Biotin BV421 DAPI-A"]].sort_values("Biotin BV421 DAPI-A")

In [ ]:
# [250121 cell 189]
del adata_qc_173.var["Gene Name"]
adata_qc_173.var

In [ ]:
# [250121 cell 190]
adata_qc_173.obs["Biotin Condition"] = adata_qc_173.obs["Biotin"].astype(str)+"_"+adata_qc_173.obs["Condition"].astype(str)

adata_qc_173.obs["Biotin Condition"].value_counts()

In [ ]:
# [250121 cell 191]
adata_qc_173.write("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/adata_qc_173_corrected_Biotin_250121.h5ad")

### 1.1 Index sorting values for panel B
Writes `correct_biotin_173cells_for_Biotin_EPCR_plot_250228.csv`, on 173 cells.

In [ ]:
# [250228 cell 28]
adata_qc = sc.read_h5ad("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/adata_qc_173_corrected_Biotin_250121.h5ad")
adata_qc

In [ ]:
# [250228 cell 29]
adata_qc.obs["Biotin Condition"].value_counts()

In [ ]:
# [250228 cell 30]
adata_qc_selected_cells = adata_qc.copy()

In [ ]:
# [250228 cell 31]
adata_qc_selected_cells.var_names = adata_qc_selected_cells.var_names.astype(str)
adata_qc_selected_cells.var_names_make_unique()

In [ ]:
# [250228 cell 35]
df_tmp = adata_qc_selected_cells.obs[["Biotin Condition","Biotin BV421 DAPI-A","EPCR PE PE-A"]]

df_tmp["Biotin BV421 DAPI-A"] = np.log1p(df_tmp["Biotin BV421 DAPI-A"])
df_tmp["EPCR PE PE-A"] = np.log1p(df_tmp["EPCR PE PE-A"])

df_tmp

In [ ]:
# [250228 cell 36]
dfs = pd.melt(df_tmp,id_vars="Biotin Condition",value_vars=["Biotin BV421 DAPI-A","EPCR PE PE-A"],ignore_index=False)\
.rename(columns={"variable":"Class","value":"score"})
dfs

In [ ]:
# [250228 cell 37]
dfs.to_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/correct_biotin_173cells_for_Biotin_EPCR_plot_250228.csv")

### 1.2 The 165 cell object and the UMAP
Drops the 8 LO_Control cells from 173 cells, which leaves 165 cells.
Writes `adata_umap_165cells_250303.h5ad` and `umap_coordinates_250303.csv`.

In [ ]:
# [250228 cell 71]
adata_qc_selected_cells = sc.read_h5ad("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/adata_qc_173_corrected_Biotin_250121.h5ad")
adata_qc_selected_cells.shape

In [ ]:
# [250228 cell 72]
adata_qc_selected_cells.obs["Biotin Condition"].value_counts()

In [ ]:
# [250228 cell 73]
adata_qc_selected_cells = adata_qc_selected_cells[adata_qc_selected_cells.obs["Biotin Condition"]!="LO_Control"].copy()

In [ ]:
# [250228 cell 74]
sc.pp.filter_genes(adata_qc_selected_cells,min_counts=1)

In [ ]:
# [250228 cell 75]
adata_qc_selected_cells.shape

In [ ]:
# [250228 cell 77]
smqpp.normalise_data(adata_qc_selected_cells)

In [ ]:
# [250228 cell 80]
import warnings
warnings.filterwarnings('ignore')

def select_HVGs(adata_norm):#HVGs
    adata = adata_norm.copy()
    adata.var_names = adata.var_names.astype(str)
    adata.var_names_make_unique()
    #smqpp
    smqpp.tech_var(adata,useERCC=False,meanForFit=10)
    smqpp.plot_tech_var(adata)
    adata_hvg = adata[:,adata.uns["varGenes"]["genes"]["highVar"]]
    adata_hvg.raw = adata_norm###Store the original total_Norm&log1p matrix for scores calculation
    return adata_hvg

def run_umap3(adata_hvg,n_pcs,n_neighbors,resol):
    adata = adata_hvg.copy()
    sc.pp.scale(adata)
    sc.tl.pca(adata,svd_solver="arpack")#,n_comps=n_pcs)
    sc.pl.pca_variance_ratio(adata,log=True,n_pcs=30)
    sc.pp.neighbors(adata,n_pcs=n_pcs,n_neighbors=n_neighbors)
    sc.tl.umap(adata)
    sc.tl.leiden(adata,resolution=resol)
    #adata.obs["Categories"] = adata.obs["ESLAM"].astype(str)+"_"+adata.obs["BC"].astype(str)
    sc.pl.umap(adata,size=200,ncols=2,color=["ESLAM","Biotin Condition","leiden"],
               legend_fontsize=7,wspace=0.3)
    return adata

In [ ]:
# [250228 cell 82]
adata_hvg = select_HVGs(adata_qc_selected_cells)
print(adata_hvg)
adata_umap = run_umap3(adata_hvg,10,10,0.6)

In [ ]:
# [250228 cell 84]
adata_umap.write("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/adata_umap_165cells_250303.h5ad")

In [ ]:
# [250228 cell 94]
import pandas as pd

# Extract UMAP coordinates and metadata
umap_data = adata_umap.obsm["X_umap"]  # Extract UMAP embeddings
umap_df = pd.DataFrame(umap_data, columns=["UMAP1", "UMAP2"])  # Convert to DataFrame

# Add metadata columns (e.g., Biotin Condition)
umap_df["Biotin_Condition"] = adata_umap.obs["Biotin Condition"].values

# Save the extracted data to CSV
umap_df.to_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/umap_coordinates_250303.csv", index=False)
umap_df

## Figure 4A, table of cell numbers and average genes per cell
Writes `Fig_4A_table_250401.csv`.

In [ ]:
# [250331 cell 12]
adata_qc = sc.read_h5ad("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/adata_qc_173_corrected_Biotin_250121.h5ad")
adata_qc

In [ ]:
# [250331 cell 13]
adata_qc.obs["Biotin Condition"].value_counts()

In [ ]:
# [250331 cell 14]
adata_qc_selected_cells = adata_qc.copy()
adata_qc_selected_cells.var_names = adata_qc_selected_cells.var_names.astype(str)
adata_qc_selected_cells.var_names_make_unique()

In [ ]:
# [250331 cell 15]
adata_qc_selected_cells.obs = adata_qc_selected_cells.obs.rename(columns={"Biotin Condition":"BC"})

In [ ]:
# [250331 cell 17]
adata_qc_selected_cells.obs.groupby("BC")["n_genes"].mean()

In [ ]:
# [250331 cell 18]
adata_qc_selected_cells.obs["BC"].value_counts()

In [ ]:
# [250331 cell 20]
tmp1 = pd.DataFrame(adata_qc_selected_cells.obs["BC"].value_counts())
tmp1

In [ ]:
# [250331 cell 21]
tmp2 = pd.DataFrame(adata_qc_selected_cells.obs.groupby("BC")["n_genes"].mean())
tmp2

In [ ]:
# [250331 cell 23]
df_table1 = pd.concat([tmp1,tmp2],axis=1)
df_table1["Condition"] = pd.Series(df_table1.index).str.split("_",expand=True)[1].values
df_table1

In [ ]:
# [250331 cell 24]
df_table1["n_genes"] = df_table1["n_genes"].round().astype(int)
df_table1

In [ ]:
# [250331 cell 26]
df_table1["BC"] = df_table1.index
df_table1

In [ ]:
# [250331 cell 27]
df_table1["label"] = df_table1["BC"].map(label_map)
df_table1

In [ ]:
# [250331 cell 30]
label_map2 = {###remove the italicization
    "HI": 'Biotinᴴⁱ', 
    "LO": 'Biotinᴸᵒ'
    # Add more mappings as needed
}

In [ ]:
# [250331 cell 31]
df_table1["Biotin"] = df_table1["BC"].str.split("_",expand=True)[0].values

In [ ]:
# [250331 cell 32]
df_table1["Biotin"] = df_table1["Biotin"].map(label_map2)
df_table1

In [ ]:
# [250331 cell 33]
df_table1 = df_table1.rename(columns={"count":"#Cells","n_genes":"Avg genes/cell"})
df_table1

In [ ]:
# [250331 cell 35]
df_table1[["Biotin","Condition","#Cells","Avg genes/cell"]]

In [ ]:
# [250331 cell 36]
df_table1[["Biotin","Condition","#Cells","Avg genes/cell"]].to_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_4A_table_250401.csv",index=False)

## Figure 4B, violin plots of Biotin BV421 and EPCR PE
Writes `Fig_4B_violin_plot_Biotin_250429.pdf` and `Fig_4B_violin_plot_EPCR_250429.pdf`.

In [ ]:
# [250331 cell 43]
dfs = pd.read_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/correct_biotin_173cells_for_Biotin_EPCR_plot_250228.csv",index_col=0)
dfs

In [ ]:
# [250331 cell 44]
dfs["Class"] = dfs["Class"].map({"Biotin BV421 DAPI-A":"Biotin BV421","EPCR PE PE-A":"EPCR PE"})
dfs = dfs.rename(columns={"Biotin Condition":"BC"})

In [ ]:
# [250331 cell 45]
dfs_biotin = dfs[dfs["Class"]=="Biotin BV421"].copy()
dfs_biotin.shape

In [ ]:
# [250331 cell 46]
dfs_epcr = dfs[dfs["Class"]=="EPCR PE"].copy()
dfs_epcr.shape

In [ ]:
# [250331 cell 49]
%%R -i dfs_biotin

df_1_2_v1 = dfs_biotin

library(ggplot2)
library(ggpubr)
library(dplyr)
library(tidyr)

# Assuming your data is in a dataframe called df_1_2_v1
# If not, you'll need to load your data first

HI_Control_color = "#015493"
HI_Infected_color = "#941914"
LO_Control_color = "#42257e"
LO_Infected_color = "#009193" 



# Define colors
HI_Control_color = "#015493"
HI_Infected_color = "#941914"
LO_Control_color = "#42257e"
LO_Infected_color = "#009193" 


# Define colors
d_colors <- c(
  "HI_Control" = HI_Control_color,
  "HI_Infected" = HI_Infected_color,
  "LO_Control" = LO_Control_color,
  "LO_Infected" = LO_Infected_color
)
df_1_2_v1$BC <- factor(df_1_2_v1$BC, levels = c("HI_Control","HI_Infected","LO_Infected"))
# Your existing ggplot code
p <- ggplot(df_1_2_v1, aes(x = `BC`, y = `score`, fill = `BC`,color=`BC`)) +
  geom_jitter(width = 0.1, size = 0.8, alpha = 0.6, color = "black") +
  geom_violin(scale = "width", size=0.8, alpha = 0.2, width = 0.7, adjust = 1, trim = FALSE) +
  geom_boxplot(width = 0.15,color="white",alpha = 0.6,outlier.shape=NA) +#, color="white"
  #geom_hline(data = median_values, aes(yintercept = median_score),linetype = "dashed", color = "black", alpha = 0.7) +
  scale_fill_manual(values = d_colors,guide = "none") +  # Fill colors with transparency
  scale_color_manual(values = d_colors, guide = "none") +

  scale_x_discrete(limits = c("HI_Control", "HI_Infected", "LO_Infected"),
                   labels = c(
                     expression(atop("Control", Biotin^Hi)),#Biotin^italic(Hi)
                     expression(atop("Infected", Biotin^Hi)),
                     expression(atop("Infected", Biotin^Lo))
                   ))+

theme_classic()+
  theme(
    legend.position = "none",
    axis.title.x = element_blank(),
    axis.text.x = element_text(size = 13, angle = 0, hjust = 0.5),
    #axis.text.x = element_blank(),
    #axis.ticks.x = element_blank(),
    axis.text.y = element_text(size = 12),
    axis.title.y = element_text(size = 13),
    #panel.grid.major = element_blank(),
    #panel.grid.minor = element_blank()
  ) +
  labs(y = paste("Log-Normalised Fluorescence Units","\n","Biotin BV421")) #+

y_max <- max(df_1_2_v1$score)

p1 <- p + geom_pwc(
    aes(group = `BC`),tip.length = 0, method = "wilcox_test", label = "p",bracket.nudge.y = 0.18,step.increase=0.1

  )+
  coord_cartesian(ylim = c(NA, y_max + 1.2), clip = "off") +  # extra head-room
  theme(plot.margin = margin(t = 8, r = 5, b = 5, l = 5))#+
  #labs(y = "log-normalized expression")

print(p1)

# Save the plot
ggsave("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_4B_violin_plot_Biotin_250429.pdf", p1, width = 3.5, height = 3.5, dpi = 500)

In [ ]:
# [250331 cell 50]
%%R -i dfs_epcr

df_1_2_v1 = dfs_epcr

library(ggplot2)
library(ggpubr)
library(dplyr)
library(tidyr)

# Assuming your data is in a dataframe called df_1_2_v1
# If not, you'll need to load your data first

HI_Control_color = "#015493"
HI_Infected_color = "#941914"
LO_Control_color = "#42257e"
LO_Infected_color = "#009193" 



# Define colors
HI_Control_color = "#015493"
HI_Infected_color = "#941914"
LO_Control_color = "#42257e"
LO_Infected_color = "#009193" 


# Define colors
d_colors <- c(
  "HI_Control" = HI_Control_color,
  "HI_Infected" = HI_Infected_color,
  "LO_Control" = LO_Control_color,
  "LO_Infected" = LO_Infected_color
)

# Your existing ggplot code

df_1_2_v1$BC <- factor(df_1_2_v1$BC, levels = c("HI_Control","HI_Infected","LO_Infected"))

p <- ggplot(df_1_2_v1, aes(x = `BC`, y = `score`, fill = `BC`,color=`BC`)) +
  geom_jitter(width = 0.1, size = 0.8, alpha = 0.6, color = "black") +
  geom_violin(scale = "width", size=0.8, alpha = 0.2, width = 0.7, adjust = 1, trim = FALSE) +
  geom_boxplot(width = 0.15,color="white",alpha = 0.6,outlier.shape=NA) +#, color="white"
  #geom_hline(data = median_values, aes(yintercept = median_score),linetype = "dashed", color = "black", alpha = 0.7) +
  scale_fill_manual(values = d_colors,guide = "none") +  # Fill colors with transparency
  scale_color_manual(values = d_colors, guide = "none") +
  scale_x_discrete(limits = c("HI_Control", "HI_Infected", "LO_Infected"),
                   labels = c(
                     expression(atop("Control", Biotin^Hi)),#Biotin^italic(Hi)
                     expression(atop("Infected", Biotin^Hi)),
                     expression(atop("Infected", Biotin^Lo))
                   )) +
# Create the plot
# p <- ggplot(df_1_2_v1, aes(x = `BC`, y = `MHC_Class2_Core_Enrichment`, fill = `BC`)) +
  # geom_violin(scale = "width", alpha = 0.9, width = 0.65,adjust = 1,trim=FALSE) +
  # #geom_dotplot(binaxis='y', stackdir='center', dotsize=0.6)+
  # #coord_flip() +
  # #geom_point() +
  # geom_jitter(width = 0.1, size = 0.8, alpha = 0.8, color = "black") +
  # geom_boxplot(width = 0.1, color = 'white', alpha = 0.2) +
  #scale_fill_manual(values = d_colors) +
  #scale_x_discrete(limits = c("HI_Control", "HI_Infected"),
  #                 labels = c("Control\nBiotin^Hi", "Infected\nBiotin^Hi")) +
#scale_y_continuous(breaks=c(-3,-2,-1,0,1,2,3,4,5),labels=c("-3","-2","-1","0","1","2","3","4","5"))+
theme_classic()+
  theme(
    legend.position = "none",
    axis.title.x = element_blank(),
    axis.text.x = element_text(size = 13, angle = 0, hjust = 0.5),
    #axis.text.x = element_blank(),
    #axis.ticks.x = element_blank(),
    axis.text.y = element_text(size = 12),
    axis.title.y = element_text(size = 13),
    #panel.grid.major = element_blank(),
    #panel.grid.minor = element_blank()
  ) +
  #scale_y_continuous(breaks=c(2,4,6,8,10,12,14),labels=c("2","4","6","8","10","12","14"))+
  scale_y_continuous(
    breaks  = c(3, 6, 9, 12, 15),        # or seq(3, 12, by = 3)
    labels  = c("3", "6", "9", "12", "15") # labels are optional—defaults would match
  )+
  labs(y = paste("Log-Normalised Fluorescence Units","\n","EPCR PE")) #+
  #coord_cartesian(ylim = c(-2.9, 5.2), xlim = c(0.95, 2.05)) 
  #geom_hline(yintercept = median(df_1_2_v1$`MHC_Class2_Core_Enrichment`), linetype = "dashed", color = "black", alpha = 0.7)

# Add statistical annotations
#p <- p + stat_compare_means(
#  comparisons = list(c("HI_Infected", "HI_Control")),
#  method = "wilcox.test",
#  label = "p = {p}",#"p.signif",
#  label.y = c(3.1, 3.2),size=4.5
#)
#

y_max <- max(df_1_2_v1$score)

p1 <- p + geom_pwc(
    aes(group = `BC`),tip.length = 0, method = "wilcox_test", label = "p",bracket.nudge.y = 0.29,step.increase=0.12

  )+
  coord_cartesian(ylim = c(NA, y_max + 3.7), clip = "off") +  # extra head-room
  theme(plot.margin = margin(t = 8, r = 5, b = 5, l = 5))#+
  #labs(y = "log-normalized expression")

print(p1)


#p <- p+annotate("text", x = 2.3, y = 1, 
#             label =  paste("Cd74","H2-Aa","H2-Eb1","H2-Ab1","Ctss","Tubb1",
#"Ctsk",
#"H2-DMa",
#"H2-DMb1",
#"H2-0a",sep="\n"),
#             hjust = 0, vjust = 0.5, size = 4, color = "red")

# Print the plot


# Save the plot
ggsave("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_4B_violin_plot_EPCR_250429.pdf", p1, width = 3.5, height = 3.5, dpi = 500)

## Figure 4C, UMAP
Writes `Fig_4C_UMAP_165_250429.pdf`.

In [ ]:
# [250331 cell 87]
%%R

# Load Required Libraries
library(ggplot2)
library(dplyr)
library(readr)

# Load UMAP coordinates extracted from Python
umap_data <- read_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/umap_coordinates_250303.csv")

# Define Colors for Biotin Conditions


HI_Control_color = "#015493"
HI_Infected_color = "#941914"
LO_Control_color = "#42257e"
LO_Infected_color = "#009193" 


# Define colors
d_colors <- c(
  "HI_Control" = HI_Control_color,
  "HI_Infected" = HI_Infected_color,
  "LO_Control" = LO_Control_color,
  "LO_Infected" = LO_Infected_color
)

legend_labels <- c(
  expression("Control\nBiotin"^Hi),
  expression("Infected  \nBiotin^Hi"),
  expression("Control\nBiotin^Lo"),
  expression("Infected\nBiotin^Lo")
)

legend_labels <- c(
  bquote("Control"~"\nBiotin"^Hi),
  bquote("Infected"~"\nBiotin"^Hi),
  bquote("Control"~"\nBiotin"^Lo),
  bquote("Infected"~"\nBiotin"^Lo)
)

legend_labels <- c(
  bquote(atop("Control", Biotin^Hi)),
  bquote(atop("Infected", Biotin^Hi)),
  bquote(atop("Control", Biotin^Lo)),
  bquote(atop("Infected", Biotin^Lo))
)

legend_labels <- c(
  bquote(atop("Control", paste("Biotin"^Hi))),
  bquote(atop("Infected", paste("Biotin"^Hi))),
  bquote(atop("Infected", paste("Biotin"^Lo))),
  bquote(atop("Infected", paste("Biotin"^Lo)))
)


# Convert Biotin Condition to Factor to Maintain Order
umap_data$Biotin_Condition <- factor(umap_data$Biotin_Condition, levels = names(d_colors))

# Create UMAP Dot Plot with Corrected Legend Labels and Superscripts
p <- ggplot(umap_data, aes(x = UMAP1, y = UMAP2, color = Biotin_Condition,fill = Biotin_Condition)) +
  #geom_point(size = 3, stroke = 0.5, aes(fill = Biotin_Condition), shape = 21, alpha = 0.7) +  # Fill with alpha but solid edge

  geom_point(alpha = 0.6, size = 3, shape = 21, stroke = 0.5) +  # Use dots with transparency
  scale_color_manual(values = d_colors, 
                     labels = legend_labels) +  # Apply colors & superscript labels
  scale_fill_manual(values = d_colors,labels = legend_labels)+
theme_bw(base_size= 11)+
  theme(
    legend.title = element_blank(),
    legend.text = element_text(size = 12,lineheight = 0.1),
    legend.position = "right",
    #legend.spacing.y = unit(1.5, 'cm'),         # Increase vertical space between legend items
    legend.key.height = unit(1.5, 'cm'),        # Increase the height of each legend key
    axis.text = element_blank(),  # Remove x & y tick labels
    axis.ticks = element_blank(),  # Remove x & y axis ticks
    axis.title = element_text(size = 13),
    axis.text.y = element_blank(),
    axis.title.y = element_text(size = 13),
   panel.grid.major = element_blank(),  # Remove major grid lines
   panel.grid.minor = element_blank()
    #panel.grid.major = element_blank(),
    #panel.grid.minor = element_blank()
  ) +
  labs(x = "UMAP1", y = "UMAP2", color = "", fill = "")

# Display the Plot (Do Not Save)
print(p)


#ggsave("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_4C_UMAP_165_250426.pdf", p, width = 4.4, height = 3.2, dpi = 500)
#ggsave("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_4C_UMAP_165_250426.pdf", p, width = 4.2, height = 3, dpi = 500)

ggsave("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_4C_UMAP_165_250429.pdf", p, width = 4.2, height = 3, dpi = 500)

## Figure 4D, cell cycle phase proportions
Writes `Fig_4D_cell_cycle_165cells_250402.pdf`.

In [ ]:
# [250331 cell 137]
cell_cycle_genes = np.genfromtxt(cc_genes_txt,dtype='str')
print(cell_cycle_genes)

def run_cell_cycle1(file):
    #adata1 = adata.copy()
    adata = sc.read(file)
    #raw data added after Norm
    adata1 = anndata.AnnData(X=adata.raw.X,obs=adata.obs,var=adata.raw.var,obsm=adata.obsm,uns=adata.uns).copy()
    print("#adata_Norm_log1p_shape:",adata1.shape)
    print("Max value in X:",adata1.X.max())
    adata1.var_names = adata1.var_names.astype(str)
    adata1.var_names_make_unique()
    print(adata1)
    #sc.pp.scale(adata1)
    cell_cycle_genes = np.genfromtxt(cc_genes_txt,dtype='str')
    print(len(cell_cycle_genes))
    print(cell_cycle_genes[:5])
    s_genes = cell_cycle_genes[:43]
    g2m_genes = cell_cycle_genes[43:]
    print(adata1.var_names[:5])
    print("Max value in X:",adata1.X.max())
    sc.tl.score_genes_cell_cycle(adata1,s_genes=s_genes,g2m_genes=g2m_genes)
    adata1_cc = adata1[:,np.intersect1d(adata1.var_names,cell_cycle_genes)]
    print("#adata_CC_shape:",adata1_cc.shape)
    sc.pp.scale(adata1)
    sc.tl.pca(adata1_cc)
    sc.pl.pca_scatter(adata1_cc,size=100,color="phase",palette='tab10')
    adata.obs['phase'] = adata1.obs['phase']
    adata.obs['G2M_score'] = adata1.obs['G2M_score']
    adata.obs['S_score'] = adata1.obs['S_score']
    return adata

def plot_cc_proportion2(adata):##plot using BC
    tab = pd.crosstab(adata.obs["Biotin Condition"],adata.obs.phase,normalize=0)
    tab = tab[["G1","S","G2M"]]
    print(tab)
    #tab.drop("LO_Control",inplace=True)
    n_classes=tab.shape[0]
    from operator import add
    fig,ax=plt.subplots(1,1,figsize=(4,4))
    bt=[0]*n_classes
    print(bt)
    for name,color in zip(tab.columns,["tab:blue","tab:green","tab:orange"]):
        print(name,color)
        ax.bar(list(tab.index),tab[name].to_list(),bottom=bt,color=color,width=0.5,label=name)
        bt=list(map(add,bt,tab[name].to_list()))
        print(bt)
    xticklabels = ax.get_xticklabels()
    ax.set_xticklabels(xticklabels,rotation=45,ha='right',rotation_mode='anchor')
    ax.legend(bbox_to_anchor=(1,1),fontsize=10)
    ax.tick_params(axis='x',labelsize=12)
    ax.tick_params(axis='y',labelsize=12)
    ax.set_ylabel("Proportion")
    ax.set_ylim(0,1.03)
    plt.grid(False)

adata_norm_hvg_umap_cc = run_cell_cycle1("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/adata_umap_165cells_250303.h5ad")
plot_cc_proportion2(adata_norm_hvg_umap_cc)

In [ ]:
# [250331 cell 138]
adata_norm_hvg_umap_cc.obs["BC"] = adata_norm_hvg_umap_cc.obs["Biotin Condition"].copy()

df_cc_plot = adata_norm_hvg_umap_cc.obs[["BC","phase"]].copy()
df_cc_plot["phase"] = df_cc_plot["phase"].replace("G1","G0/G1")
df_cc_plot

In [ ]:
# [250331 cell 140]
tab = pd.crosstab(df_cc_plot.BC,df_cc_plot.phase,normalize=0)
tab = tab[["G0/G1","S","G2M"]]
tab_long = tab.stack().reset_index()
tab_long = tab_long.rename(columns={0:"proportion"})
tab_long

In [ ]:
# [250331 cell 141]
%%R -i tab_long

library(ggplot2)
library(dplyr)
library(tidyr)
library(scales)
library(tibble)

plot_cc_proportion3 <- function(tab_long) {
  ftsize <- 13
  print(tab_long)
  #tab_long$phase <- factor(tab_long$phase, levels = c("G0/G1", "S", "G2M"))
  #tab_long$phase <- factor(tab_long$phase, levels = c( "G2M","S","G0/G1"))
 
  # Define colors
  colors <- c("#1B9E77", "#D95F02", "#7570B3")
  
  # Create label mapping
  label_map <- c(
    "HI_Control" = bquote('Control\nBiotin'^"Hi"),
    "HI_Infected" = expression("Infected\nBiotin^Hi"),

    "LO_Infected" = expression("Infected\nBiotin"^Lo)
  )
  label_map <- c(bquote(atop(Control,Biotin^"Hi")),bquote(atop(Infected,Biotin^"Hi")),bquote(atop(Infected,Biotin^"Lo")))
  # Create the plot
  p <- ggplot(tab_long, aes(x = BC, y = proportion, fill = phase)) +
    geom_bar(stat = "identity", width = 0.7,alpha=0.9, color = "white") +
    scale_fill_manual(values = colors) +
    scale_y_continuous(limits = c(0, 1.05),breaks=seq(0,1,by=0.25), expand = c(0, 0)) +
    scale_x_discrete(labels = label_map) +
    labs(y = "Proportion", x = NULL) +
    #theme_minimal() +
    theme_classic()+
    theme(
      legend.background = element_rect(fill = NA, color = NA),
      legend.position = c(1.08, 0.5),
      legend.title = element_blank(),
      axis.text.x = element_text(angle = 0, hjust = 0.5,vjust=0, size = ftsize),
      axis.text.y = element_text(size = ftsize),
      axis.title.y = element_text(size = ftsize),
      axis.title.x = element_text(size = ftsize),
      legend.text = element_text(size = 11,margin = margin(l = -5, unit = "pt")),
      panel.grid = element_blank(),
      axis.line = element_line(color = "black"),
      plot.margin = margin(5.5, 40, 5.5, 5.5)
    )
  
  print(p)
  ggsave("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_4D_cell_cycle_165cells_250402.pdf",p,dpi=500,width=3.5,height=3)
}

# Assuming 'df' is your dataframe
plot_cc_proportion3(tab_long)


## Figure 4E, dot plot of the top 20 genes per group

ranks the genes with `sc.tl.rank_genes_groups`, using a t-test on all
genes, then draws the top 20 genes per group with
`sc.pl.rank_genes_groups_dotplot`. Writes `Fig_4E_dotplot_250407.pdf`.


In [ ]:
# [250331 cell 358]
adata_qc_selected_cells = sc.read_h5ad("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/adata_qc_173_corrected_Biotin_250121.h5ad")
adata_qc_165 = adata_qc_selected_cells[adata_qc_selected_cells.obs["Biotin Condition"]!="LO_Control"].copy()
sc.pp.filter_genes(adata_qc_165,min_cells=1)
adata_qc_165.obs["BC"] = adata_qc_165.obs["Biotin Condition"].copy()

adata_qc_165.var_names = adata_qc_165.var_names.astype(str)
adata_qc_165.var_names_make_unique()

print(adata_qc_165.X.max())
adata_qc_165_bf_norm = adata_qc_165.copy()
smqpp.normalise_data(adata_qc_165)

adata_qc_165.X.max()

In [ ]:
# [250331 cell 309]
sc.set_figure_params(scanpy=True, fontsize=14)


# Define the new tick labels
new_labels = {
"HI_Control":'Control\nBiotin$^{Hi}$',
"HI_Infected":'Infected\nBiotin$^{Hi}$',
"LO_Control":'Control\nBiotin$^{Lo}$',
"LO_Infected":'Infected\nBiotin$^{Lo}$'
    # Add more mappings as needed
}

new_labels =  {
    "HI_Control": r'Control' + "\n" + r'Biotin$\rm^{Hi}$',
    "HI_Infected": r'Infected' + "\n" + r'Biotin$\rm^{Hi}$',
    "LO_Control": r'Control' + "\n" + r'Biotin$\rm^{Lo}$',
    "LO_Infected": r'Infected' + "\n" + r'Biotin$\rm^{Lo}$'
}



print(adata_qc_165.X.max())

adata_qc_171_tmp = adata_qc_165.copy()

adata_qc_171_tmp.obs["BC"] = adata_qc_171_tmp.obs["BC"].replace(new_labels)

print(adata_qc_171_tmp.obs["BC"].value_counts())

adata_qc_171_tmp.obs["BC"] = pd.Categorical(adata_qc_171_tmp.obs["BC"])

sc.tl.rank_genes_groups(adata_qc_171_tmp,groupby='BC',method='t-test',
                        n_genes=adata_qc_171_tmp.shape[1],use_raw=False,key_added="DEG_Condition")

dp = sc.pl.rank_genes_groups_dotplot(adata_qc_171_tmp, n_genes=20, key="DEG_Condition", groupby="BC",standard_scale='var',
                                swap_axes=False,return_fig=True,figsize=(15.5,2.5),show=False)

ax = dp.get_axes()["mainplot_ax"]




# Change x-axis ticklabel size and rotation
ax.set_xticklabels(ax.get_xticklabels(), fontsize=14) #rotation=90,rotation_mode='anchor',ha="center")

# Change y-axis ticklabel size
ax.set_yticklabels(ax.get_yticklabels(), fontsize=14.5)


ax_size_legend = dp.get_axes()["size_legend_ax"]
ax_size_legend.tick_params(axis='both', which='major', labelsize=14)
ax_size_legend.set_position([0.87,0.7,0.13,0.2])#left, bottom, width, height
ax_size_legend.set_title(ax_size_legend.get_title(), fontsize=15,x=0.5, y=0.5,pad=0.1)



ax_color_legend = dp.get_axes()["color_legend_ax"]
ax_color_legend.tick_params(axis='both', which='major', labelsize=14)
ax_color_legend.set_position([0.905, 0.25, 0.06, 0.04])#left, bottom, width, height
ax_color_legend.set_title(ax_color_legend.get_title(), fontsize=15,x=0.5, y=1)
# Adjust layout to prevent clipping of rotated labels
#plt.tight_layout()


l_interferon = ['Cd74',
'Irf7',
'H2-nnn',
'B2m',
'Ly6a',
'Stat1']

for label in ax.get_xticklabels():
    label.set_rotation(90)  # Rotate labels
    label.set_fontsize(14)  # Adjust font size
    label_text = label.get_text()
    if (label_text in l_interferon) or label_text.startswith("H2-"):
        print(label_text)
        label.set_color("red")

# **Rotate dendrogram labels (Forced)**
if "row_dendrogram_ax" in dp.get_axes():
    ax_dendrogram = dp.get_axes()["row_dendrogram_ax"]
    for label in ax_dendrogram.get_xticklabels():
        label.set_rotation(30)  # Rotate labels
        label.set_fontsize(15)  # Adjust font size
        label_text = label.get_text()
        print(label_text)
        if (label_text in l_interferon) or label_text.startswith("H2-"):
            label.set_color("red")
        
dp.style(cmap='Reds',
    dot_max=1,
    dot_min=0)


# Show the plot
#plt.show()

#plt.show()

plt.savefig('/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_4E_dotplot_250407.pdf', dpi=400, bbox_inches='tight')

## Supplemental Figure 5B, the same dot plot with 50 genes per group
that the cell above computed. Writes `Fig_4E_dotplot_50_Genes_250407.pdf`.

In [ ]:
# [250331 cell 310]
dp = sc.pl.rank_genes_groups_dotplot(adata_qc_171_tmp, n_genes=50, key="DEG_Condition", groupby="BC",standard_scale='var',
                                swap_axes=False,return_fig=True,figsize=(32,4),show=False)

ax = dp.get_axes()["mainplot_ax"]




# Change x-axis ticklabel size and rotation
ax.set_xticklabels(ax.get_xticklabels(), fontsize=14) #rotation=90,rotation_mode='anchor',ha="center")

# Change y-axis ticklabel size
ax.set_yticklabels(ax.get_yticklabels(), fontsize=14)



ax_size_legend = dp.get_axes()["size_legend_ax"]
ax_size_legend.tick_params(axis='both', which='major', labelsize=14)
ax_size_legend.set_position([0.92,0.7,0.13,0.2])#left, bottom, width, height
ax_size_legend.set_title(ax_size_legend.get_title(), fontsize=15,x=0.5, y=0.5,pad=0.1)



ax_color_legend = dp.get_axes()["color_legend_ax"]
ax_color_legend.tick_params(axis='both', which='major', labelsize=14)
ax_color_legend.set_position([0.95, 0.25, 0.06, 0.04])#left, bottom, width, height
ax_color_legend.set_title(ax_color_legend.get_title(), fontsize=15,x=0.5, y=1)
# Adjust layout to prevent clipping of rotated labels
#plt.tight_layout()


l_interferon = ['Cd74',
'Irf7',
'H2-nnn',
'B2m',
'Ly6a',
'Stat1']

for label in ax.get_xticklabels():
    label.set_rotation(90)  # Rotate labels
    label.set_fontsize(13.5)  # Adjust font size
    label_text = label.get_text()
    if (label_text in l_interferon) or label_text.startswith("H2-"):
        print(label_text)
        label.set_color("red")

        
dp.style(cmap='Reds',
    dot_max=1,
    dot_min=0)


plt.savefig('/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_4E_dotplot_50_Genes_250407.pdf', dpi=500, bbox_inches='tight')

## Figure 4F, volcano plot, Biotin-Hi Infected against Biotin-Hi Control

In [ ]:
# [250331 cell 211]
adata_qc_selected_cells = sc.read_h5ad("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/adata_qc_173_corrected_Biotin_250121.h5ad")
adata_qc_165 = adata_qc_selected_cells[adata_qc_selected_cells.obs["Biotin Condition"]!="LO_Control"].copy()
sc.pp.filter_genes(adata_qc_165,min_cells=1)
adata_qc_165.obs["BC"] = adata_qc_165.obs["Biotin Condition"].copy()

adata_qc_165.var_names = adata_qc_165.var_names.astype(str)
adata_qc_165.var_names_make_unique()

smqpp.normalise_data(adata_qc_165)

adata_qc_165.X.max()

In [ ]:
# [250331 cell 215]
adata_filter_HI_HC,df_DEGs_HI_HC,df_DEGs_HI_HC_Sig = compare_genes_v3(adata_qc_165,"HI_Infected","HI_Control","HI_vs_HC")

In [ ]:
# [250331 cell 224]
df_DEGs_HI_HC1 = df_DEGs_HI_HC.copy()
df_DEGs_HI_HC1 = df_DEGs_HI_HC1.rename(columns={"names":"gene",'logfoldchanges':"log2FoldChange"})
df_DEGs_HI_HC1.head()

In [ ]:
# [250331 cell 226]
%%R -i df_DEGs_HI_HC1

data <- df_DEGs_HI_HC1

library(ggplot2)
library(ggrepel)


#HI_Control_color = "#377eb8"
#HI_Infected_color = "#e31a1c"
#LO_Control_color = "#4eaf4a"
#LO_Infected_color= "#984ea4"

# Calculate -log10 of p-values
data$neg_log10_pvalue <- -log10(data$pvals_adj)
fdr <- -log10(0.05)

# Define significance thresholds
fc_threshold <- 1.5  # log2FoldChange threshold
p_threshold <- 0.05  # p-value threshold

# Add a column to categorize genes
data$category <- ifelse(data$log2FoldChange >= fc_threshold & data$pvals_adj < p_threshold, "Up",
                 ifelse(data$log2FoldChange <= -fc_threshold & data$pvals_adj < p_threshold, "Down",
                        "Not Significant"))

# List of genes you want to annotate  # previous one 250406
#genes_to_annotate <- c('Cd74', 'Plac8', 'Gvin3', 'Irf7', 'Gm4951', 'Serpina3g', 'Oas2', 'Serpina3f', 'Rtp4', 'Gbp7',
#                       'Rtn2', 'Gm26802', 'Nr4a1', 'Egr1', 'Maged1', 'Sik1', 'Neat1', 'Iqcn', 'Atf4', "Tgfb1","Bcl2")#, 'Gm20429')

genes_to_annotate <- c('Cd74', 'Plac8', 'Gvin3', 'Irf7', 'Gm4951', 'Serpina3f', 'Oas2', 'Serpina3g', 'Gm12185', 'H2-Aa',
                        'Egr1', 'Gtdc1', 'Gm26802', 'Maged1', 'Peg13', 'Nr4a1', 'Sik1', 'Iqcn', 'Atf4',"Tgfb1","Bcl2")

# Prepare genes to be annotated
genes_to_label <- subset(data, gene %in% genes_to_annotate)

# Create the volcano plot
volcano_plot <- ggplot(data, aes(x = log2FoldChange, y = neg_log10_pvalue)) +
  geom_point(aes(color = category), size = 1) +
  scale_color_manual(values = c("Up" = "#e31a1c", "Down" = "#377eb8", "Not Significant" = "grey")) +
  theme_minimal() +
  labs(
    title = expression(paste("Infected Biotin"^italic(Hi), " vs ", "Control Biotin"^italic(Hi))),
    x = "Log2 Fold Change",
    y = "-Log10 P-value"
  ) +
  theme(
    plot.title = element_text(size = 13, face = "bold", hjust = 0.5, margin = margin(b = 1, t = 0), color = "black"),
    legend.position = c(0.43, 1.04),  # Position legend in top right corner
    legend.justification = c(1, 1),   # Align legend to top right corner
    legend.box.just = "right",
    legend.margin = margin(6, 6, 6, 6),
    #legend.position = "right",
    legend.text = element_text(size = 12),  # Increase legend text size
    legend.title = element_blank(),  # 
    panel.grid = element_blank(),  # Remove grid
    axis.line = element_line(color = "black"),  # Add axis lines
    plot.background = element_rect(fill = "white", color = NA),  # White background
    panel.background = element_rect(fill = "white", color = NA),  # White panel
    axis.ticks = element_line(color = "black"),  # Add black ticks
    axis.ticks.length = unit(0.2, "cm"),  # Set tick length
    axis.text = element_text(size = 11),  # Increase axis text size
    axis.title = element_text(size = 12),  # Increase axis title size
    axis.text.x.top = element_blank(),  # Remove top x-axis labels
    axis.ticks.x.top = element_blank(),  # Remove top x-axis ticks
    axis.line.x.top = element_line(color = "black"),
    axis.text.y.right = element_blank(),  # Remove right y-axis labels
    axis.ticks.y.right = element_blank(),  # Remove right y-axis ticks
    axis.text.x = element_text(size = 14),
    axis.text.y = element_text(size = 14),
    axis.title.x = element_text(size = 14, margin = margin(t = 5)),  # Increase x-axis label size and add margin
    axis.title.y = element_text(size = 14, margin = margin(r = 5)),  # Increase y-axis label size and add margin
  )


# Add threshold lines
volcano_plot <- volcano_plot +
  geom_hline(yintercept = -log10(p_threshold), linetype = "dashed",color = "grey50") +
  geom_vline(xintercept = c(-fc_threshold, fc_threshold), linetype = "dashed", color = "grey50")

fc <- 25
mo <- 20
# Add text labels for upregulated genes
volcano_plot <- volcano_plot +
  geom_text_repel(
    data = subset(genes_to_label, category == "Up"),
    aes(label = gene),
    color = "#e31a1c",
    size = 4,  # Increase text size
    box.padding = 0.5,
    point.padding = 0.5,
    force = fc,  # Increase force to spread labels more
    nudge_x = 1,
    direction = "y",
    hjust = 0,
    segment.size = 0.3,
    segment.color = "grey50",
    min.segment.length = 0,
    max.overlaps = mo,  # Limit overlaps
    show.legend = FALSE
  )

# Add text labels for downregulated genes
volcano_plot <- volcano_plot +
  geom_text_repel(
    data = subset(genes_to_label, category == "Down"),
    aes(label = gene),
    color = "#377eb8",
    size = 4,  # Increase text size
    box.padding = 0.5,
    point.padding = 0.5,
    force = fc+15,  # Increase force to spread labels more
    nudge_x = -1,
    direction = "y",
    hjust = 1,
    segment.size = 0.3,
    segment.color = "grey50",
    min.segment.length = 0,
    max.overlaps = mo,  # Limit overlaps
    show.legend = FALSE
  )

volcano_plot <- volcano_plot +
  geom_text_repel(
    data = subset(genes_to_label, category == "Not Significant"),
    aes(label = gene),
    color = "dimgrey",
    size = 4,  # Increase text size
    box.padding = 0.5,
    point.padding = 0.5,
    force = 20,  # Increase force to spread labels more
    nudge_x = 1,
    direction = "y",
    hjust = 0,
    segment.size = 0.3,
    segment.color = "grey50",
    min.segment.length = 0,
    max.overlaps = 20,  # Limit overlaps
    show.legend = FALSE
  )

# Set specific x and y axis limits
volcano_plot <- volcano_plot +
  coord_cartesian(
    xlim = c(-12, 12),  # Set x-axis limits
    ylim = c(0, 16)   # Set y-axis limits
  ) +
  scale_x_continuous(breaks = seq(-12, 12, by = 4),sec.axis = dup_axis(name = NULL) ) +  # Set x-axis breaks
  scale_y_continuous(breaks = seq(0, 16, by = 4),minor_breaks = seq(0, 12, by = 4),sec.axis = dup_axis(name = NULL))    # Set y-axis breaks

# Display the plot
print(volcano_plot)

# Save the plot
ggsave("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/volcano_plot_annotated_HI_HC_250402.pdf", volcano_plot, width = 4.2, height = 4.2, dpi = 500)


## Figure 4G, Gene Ontology bar plot
The ranked gene file is written first, Enrichr runs outside the notebook,
and the Enrichr table is read back.

In [ ]:
# [250331 cell 229]
def generate_pre_rank_file(df_deg_res,opt_file):
    df_deg_res = df_deg_res.copy()
    df_deg_res["Rank"] = -np.log10(df_deg_res["pvals_adj"])*df_deg_res["logfoldchanges"]
    df_deg_res = df_deg_res.sort_values("Rank",ascending=False).reset_index(drop=True)
    ranking = df_deg_res[["names","Rank"]]
    ranking.to_csv(opt_file,sep="\t",header=False,index=False)
    return ranking

In [ ]:
# [250331 cell 230]
df_rank = generate_pre_rank_file(df_DEGs_HI_HC,"/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/HI_HC_pre_rank_250407.rnk")
df_rank

In [ ]:
# [250331 cell 263]
df_GO_pvals = pd.read_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/GO_Biological_Process_2025_table_HI_HC_250407.txt",sep="\t",header=0)
df_GO_pvals.head()

In [ ]:
# [250331 cell 264]
%%R -i df_GO_pvals


library(ggplot2)
library(dplyr)
library(scales)

# Read the data

# Function to create the enrichment figure
enrichr_figure <- function(df_GO_pvals, plot_name, all_libraries, bar_color) {

  # Prepare the data
  # Select top 10 rows if needed
  df_GO_pvals <- df_GO_pvals %>% head(5)

  df_GO_pvals$log10_pvalue <- -log10(df_GO_pvals$`Adjusted P-value`)
  df_GO_pvals$Term <- factor(df_GO_pvals$Term, levels = rev(df_GO_pvals$Term))
  
  # Set bar colors
  df_GO_pvals$bar_color <- ifelse(df_GO_pvals$`Adjusted P-value` < 0.05, bar_color, "lightgrey")
  
  # Create the plot
  p <- ggplot(df_GO_pvals, aes(x = log10_pvalue, y = Term, fill = bar_color)) +
    geom_bar(stat = "identity", alpha = 0.5) +
    scale_fill_identity() +
    theme_minimal() +
    theme(
      axis.text.y = element_blank(),
      axis.ticks.y = element_blank(),
      panel.grid = element_blank(),
      plot.title = element_text(size = 20),
      axis.title.x = element_text(size = 20,margin = margin(r = 5)),
      axis.text.x = element_text(size = 20),
      #axis.line.x = element_line(color = "black", size = 0.5,),  # Add x-axis spine
      axis.ticks.x = element_line(color = "black", size = 0.5),  # Add this line for x-axis ticks
      axis.ticks.length = unit(0.25, "cm"),  # Adjust tick length
      #axis.line.y = element_line(color = "black", size = 0.5)   # Add y-axis spine

    ) +
    labs(x = "-log10(adjusted p-value)", y = NULL,
         title= expression(paste("Infected Biotin"^Hi, " vs ", "Control Biotin"^Hi)), ) +
    scale_x_continuous(
      limits = c(0, 15.1),
      breaks = c(0, 5, 10, 15),
      labels = c("0", "5", "10", "15")
    )+
    coord_cartesian(clip = "off")
  
  # Add text annotations

# Reverse the order for annotation
for (i in nrow(df_GO_pvals):1) {
  annot <- df_GO_pvals$Term[i]
  if (df_GO_pvals$`Adjusted P-value`[i] >= 0.05) {
    annot <- paste(annot, format(df_GO_pvals$`Adjusted P-value`[i], scientific = TRUE, digits = 2))
  }
  p <- p + annotate("text", x = 0.1, y = nrow(df_GO_pvals) - i + 1, label = annot, hjust = 0, 
                    size = 5.5)
}

  p <- p + 
    geom_segment(aes(x = 0, xend = 15.1, y = 0.1, yend = 0.1), color = "black", size = 0.5) +
    geom_segment(aes(x = 0, xend = 0, y = 0, yend = 5.5), color = "black", size = 0.5)
  # Save the plot
  ggsave(plot_name, p, width = 7, height = 3.5,dpi=500)
  
  # Show the plot
  print(p)
}

# Call the function
enrichr_library <- 'GO_Biological_Process_2025'
color <- 'cornflowerblue'
enrichr_figure(df_GO_pvals, "/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_4F_HI_vs_HC_GO_barplot_250407.pdf", enrichr_library, color)


## Figure 4H, volcano plot, Biotin-Hi Infected against Biotin-Lo Infected
use an adjusted p-value threshold of 0.1, which matches the published panel.

In [ ]:
# [250331 cell 236]
adata_filter2,df_DEGs2,df_DEGs2_Sig = compare_genes_v3(adata_qc_165,"HI_Infected","LO_Infected","HI_vs_LI")

In [ ]:
# [250331 cell 241]
df_DEGs22 = df_DEGs2.copy()
df_DEGs22 = df_DEGs22.rename(columns={"names":"gene",'logfoldchanges':"log2FoldChange"})
df_DEGs22.head()

In [ ]:
# [250331 cell 253]
%%R -i df_DEGs22

data <- df_DEGs22

library(ggplot2)
library(ggrepel)


#HI_Control_color = "#377eb8"
#HI_Infected_color = "#e31a1c"
#LO_Control_color = "#4eaf4a"
#LO_Infected_color= "#984ea4"

# Calculate -log10 of p-values
data$neg_log10_pvalue <- -log10(data$pvals_adj)

p_threshold <- 0.1  # p-value threshold
fdr <- -log10(p_threshold)

# Define significance thresholds
fc_threshold <- 1.5  # log2FoldChange threshold


# Add a column to categorize genes
data$category <- ifelse(data$log2FoldChange >= fc_threshold & data$pvals_adj < p_threshold, "Up",
                 ifelse(data$log2FoldChange <= -fc_threshold & data$pvals_adj < p_threshold, "Down",
                 ifelse(data$log2FoldChange > 0, "Not Significant Positive", "Not Significant Negative")))

# List of genes you want to annotate
#genes_to_annotate <- c('Mettl7a1', 'Cd74', 'Procr',#, 'Gm5518'
#                       'AU020206', 'Nemf', 'Rpl28', 'Bcl2', 'Rtca', 'Kat2a', 'Cryzl1','Cd9', 'Pdcd4', 'Tgfb1')#, 'Gm20429')


genes_to_annotate <- c('Mettl7a1', 'Cd74', 'Procr', 'Bcl2', 'Rtca', 'Kat2a', 'Cryzl1',
       'Cd9', 'Tgfb1')

# Prepare genes to be annotated
genes_to_label <- subset(data, gene %in% genes_to_annotate)

# Create the volcano plot
volcano_plot <- ggplot(data, aes(x = log2FoldChange, y = neg_log10_pvalue)) +
  geom_point(aes(color = category), size = 1) +
  scale_color_manual(values = c("Up" = "#e31a1c", "Down" = "#377eb8", "Not Significant Positive" = "grey","Not Significant Negative" = "grey"),
            labels = c("Up" = "Up",
               "Down" = "Down",
               "Not Significant Positive" = "Not Significant",
               "Not Significant Negative" = "Not Significant")) +
  theme_minimal() +
  labs(
    title = expression(paste("Infected Biotin"^italic(Hi), " vs ", "Infected Biotin"^italic(Lo))),
    x = "Log2 Fold Change",
    y = "-Log10 P-value"
  ) +
  theme(
    plot.title = element_text(size = 13, face = "bold", hjust = 0.5, margin = margin(b = 1, t = 0), color = "black"),
    legend.position = c(0.43, 1.04),  # Position legend in top right corner
    legend.justification = c(1, 1),   # Align legend to top right corner
    legend.box.just = "right",
    legend.margin = margin(6, 6, 6, 6),
    #legend.position = "right",
    legend.text = element_text(size = 12),  # Increase legend text size
    legend.title = element_blank(),  # 
    panel.grid = element_blank(),  # Remove grid
    axis.line = element_line(color = "black"),  # Add axis lines
    plot.background = element_rect(fill = "white", color = NA),  # White background
    panel.background = element_rect(fill = "white", color = NA),  # White panel
    axis.ticks = element_line(color = "black"),  # Add black ticks
    axis.ticks.length = unit(0.2, "cm"),  # Set tick length
    axis.text = element_text(size = 11),  # Increase axis text size
    axis.title = element_text(size = 12),  # Increase axis title size
    axis.text.x.top = element_blank(),  # Remove top x-axis labels
    axis.ticks.x.top = element_blank(),  # Remove top x-axis ticks
    axis.line.x.top = element_line(color = "black"),
    axis.text.y.right = element_blank(),  # Remove right y-axis labels
    axis.ticks.y.right = element_blank(),  # Remove right y-axis ticks
    axis.text.x = element_text(size = 14),
    axis.text.y = element_text(size = 14),
    axis.title.x = element_text(size = 14, margin = margin(t = 5)),  # Increase x-axis label size and add margin
    axis.title.y = element_text(size = 14, margin = margin(r = 5)),  # Increase y-axis label size and add margin
  )#+
  #guides(color = guide_legend(override.aes = list(color = c("#377eb8","grey","#e31a1c"))))


# Add threshold lines
volcano_plot <- volcano_plot +
  geom_hline(yintercept = -log10(p_threshold), linetype = "dashed",color = "grey50") +
  geom_vline(xintercept = c(-fc_threshold, fc_threshold), linetype = "dashed", color = "grey50")

fc <- 25
mo <- 10
# Add text labels for upregulated genes
volcano_plot <- volcano_plot +
  geom_text_repel(
    data = subset(genes_to_label, category == "Up"),
    aes(label = gene),
    color = "#e31a1c",
    size = 4,  # Increase text size
    box.padding = 0.5,
    point.padding = 0.5,
    force = fc,  # Increase force to spread labels more
    nudge_x = 1,
    direction = "y",
    hjust = 0,
    segment.size = 0.3,
    segment.color = "grey50",
    min.segment.length = 0,
    max.overlaps = mo,  # Limit overlaps
    show.legend = FALSE
  )

# Add text labels for downregulated genes
volcano_plot <- volcano_plot +
  geom_text_repel(
    data = subset(genes_to_label, category == "Down"),
    aes(label = gene),
    color = "#377eb8",
    size = 4,  # Increase text size
    box.padding = 0.5,
    point.padding = 0.5,
    force = fc+15,  # Increase force to spread labels more
    nudge_x = -1,
    direction = "y",
    hjust = 1,
    segment.size = 0.3,
    segment.color = "grey50",
    min.segment.length = 0,
    max.overlaps = 9,  # Limit overlaps
    show.legend = FALSE
  )

volcano_plot <- volcano_plot +
  geom_text_repel(
    data = subset(genes_to_label, category == "Not Significant Positive"),
    aes(label = gene),
    color = "dimgrey",
    size = 4,  # Increase text size
    box.padding = 0.5,
    point.padding = 0.5,
    force = 20,  # Increase force to spread labels more
    nudge_x = 1,
    direction = "y",
    hjust = 0.5,
    segment.size = 0.3,
    segment.color = "grey50",
    min.segment.length = 0,
    max.overlaps = 20,  # Limit overlaps
    show.legend = FALSE
  )

volcano_plot <- volcano_plot +
  geom_text_repel(
    data = subset(genes_to_label, category == "Not Significant Negative"),
    aes(label = gene),
    color = "dimgrey",
    size = 4,  # Increase text size
    box.padding = 0.5,
    point.padding = 0.5,
    force = 20,  # Increase force to spread labels more
    nudge_x = -1,
    direction = "y",
    hjust = 0.5,
    segment.size = 0.3,
    segment.color = "grey50",
    min.segment.length = 0,
    max.overlaps = 20,  # Limit overlaps
    show.legend = FALSE
  )


# Set specific x and y axis limits
volcano_plot <- volcano_plot +
  coord_cartesian(
    xlim = c(-6, 6),  # Set x-axis limits
    ylim = c(0, 2.4)   # Set y-axis limits
  ) +
  scale_x_continuous(breaks = seq(-6, 6, by = 2),sec.axis = dup_axis(name = NULL) ) +  # Set x-axis breaks
  scale_y_continuous(breaks = seq(0, 4, by = 1),minor_breaks = seq(0, 12, by = 4),sec.axis = dup_axis(name = NULL))    # Set y-axis breaks

# Display the plot
print(volcano_plot)

# Save the plot
ggsave("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/volcano_plot_annotated_HI_LI_250402.pdf", volcano_plot, width = 4.2, height = 4.2, dpi = 300)


## Figure 4I left, scFEA glycolysis and TCA cycle volcano plot

It starts from `adata_qc_selected_143cells_quantile40_250125.h5ad`, drops LO_Control,
and keeps 135 cells, which are 54 HI_Control, 38 HI_Infected and 43 LO_Infected.
The likelihood ratio test compares 92 Biotin-Hi cells against 43 Biotin-Lo cells,
so the Biotin-Hi group pools control and infected cells.
Writes `Fig_4I_TCA_volcano_plot_250205.pdf`.

In [ ]:
# [250228 cell 209]
adata_qc_selected_cells = sc.read_h5ad("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/adata_qc_selected_143cells_quantile40_250125.h5ad")

print(adata_qc_selected_cells.X.max())


In [ ]:
# [250228 cell 210]
adata_qc_selected_cells.obs = adata_qc_selected_cells.obs.rename(columns={"Biotin Condition":"BC"})

adata_qc_selected_cells = adata_qc_selected_cells[adata_qc_selected_cells.obs["BC"]!="LO_Control"].copy()
sc.pp.filter_genes(adata_qc_selected_cells,min_counts=1)

adata_qc_selected_cells.shape

In [ ]:
# [250228 cell 212]
smqpp.normalise_data(adata_qc_selected_cells)

In [ ]:
# [250228 cell 213]
print(adata_qc_selected_cells.X.max())

In [ ]:
# [250228 cell 214]
adata_qc_selected_cells.obs.BC.value_counts()

In [ ]:
# [250228 cell 306]
metab_genes = np.genfromtxt("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_0802/scFEA_analysis/scripts/data/scFEA.mouse.genes.txt",dtype=str)


In [ ]:
# [250228 cell 309]
intersected_genes = np.intersect1d(metab_genes,adata_qc_selected_cells.var_names)
print(len(intersected_genes))

adata_or2 = adata_qc_selected_cells[:,intersected_genes].copy()
adata_or2

In [ ]:
# [250228 cell 310]
from scipy.sparse import csr_matrix
pd.DataFrame(csr_matrix(adata_or2.X).todense(),index=adata_or2.obs.index,columns=adata_or2.var.index).T.to_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/SLX21561_scFEA_metab_genes_CPM_250203.csv")

df_for_scFEA = pd.read_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/SLX21561_scFEA_metab_genes_CPM_250203.csv",index_col=0,header=0)
df_for_scFEA.head()

In [ ]:
# [250228 cell 315]
adata_qc_selected_cells.obs["Cell_type_general"] = "SLAM"

In [ ]:
# [250228 cell 316]
adata_qc_selected_cells.obs.BC.value_counts()

In [ ]:
# [250228 cell 317]
adata_qc_selected_cells.obs.Biotin.value_counts()

In [ ]:
# [250228 cell 318]
adata_qc_selected_cells.obs[["Sequencing_identifier","Biotin","Cell_type_general"]].rename(columns={"Sequencing_identifier":'dataset',"Biotin":'Condition',"Cell_type_general":'celltype'})


In [ ]:
# [250228 cell 319]
adata_qc_selected_cells.obs[["Sequencing_identifier","Biotin","Cell_type_general"]].rename(columns={"Sequencing_identifier":'dataset',"Biotin":'Condition',"Cell_type_general":'celltype'}).to_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/metadata_for_LR_250203.csv")

In [ ]:
# [250228 cell 322]
# [ql321@cpu-r-4 scFEA_analysis_250203]$ sbatch run_scFEA_250203.sh

In [ ]:
# [250228 cell 323]
# (seurat_env2) [ql321@cpu-r-4 scFEA_analysis_250203]$ Rscript LR_test_Qi_Biotin_HI_vs_LO_250203.R 
# Attaching SeuratObject
# Attaching sp
# Registered S3 method overwritten by 'SeuratDisk':
  # method            from  
  # as.sparse.H5Group Seurat
# [1] "!Input Flux file:"
# [1] "!Input metadata:"
# Warning: Feature names cannot have underscores ('_'), replacing with dashes ('-')
# Analysis completed

In [ ]:
# [250228 cell 324]
df_l_test = pd.read_csv('/home/ql321/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/scFEA_analysis_250203/LR_test_250203_SLAM_metab_HI_vs_LO.csv',sep=",",index_col=0,header=0)
df_l_test.index = df_l_test.index.str.replace("-","_")
df_l_test

In [ ]:
# [250228 cell 325]
meta_new = pd.read_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/scFEA_analysis_240731/scFEA.M171.mouse.moduleinfo_AddAnno_2.csv",index_col=0)
meta_new = meta_new.rename(columns={"SM_anno":"SM_anno2"})
meta_new.head()

In [ ]:
# [250228 cell 327]
m171 = pd.read_csv("/home/ql321/ql321/work/work_SLX21561_SLX21562/analysis_0802/scFEA_analysis/scripts/data/scFEA.M171.mouse.moduleinfo.csv",sep=",",index_col=0,header=0)
m171

In [ ]:
# [250228 cell 328]
df_l_test2 = df_l_test.merge(m171,how='left',left_index=True,right_index=True)
df_l_test2

In [ ]:
# [250228 cell 331]

def sort_modules(df_l_test):
    t=df_l_test.loc[df_l_test['p_val']<0.05].sort_index()
    t['index']=t.index
    t['n']=t['index'].str.split("_",expand=True)[1]
    t["name"]=t.index
    t["n"]=t["n"].astype(int)
    lr_results = t.sort_values('n')[["p_val","p_val_adj","name"]]
    print(lr_results.head())
    return lr_results

flux_results2 = sort_modules(df_l_test2)

#Check which super module is significant
pd.merge(flux_results2,m171[['M_name','SM_anno']],left_on="name",right_index=True,how='inner')


In [ ]:
# [250228 cell 332]
df_meta = pd.read_csv('/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/scFEA_analysis_250203/metadata_for_LR_250203.csv',index_col=0)
df_meta.head()

In [ ]:
# [250228 cell 334]
flux = pd.read_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/scFEA_analysis_250203/scFEA_output_250203/Flux.csv",index_col=0)
flux.head()

In [ ]:
# [250228 cell 335]
flux.shape

In [ ]:
# [250228 cell 336]
adata_flux = anndata.AnnData(X=flux,obs=df_meta.loc[flux.index][["Condition"]])
adata_flux

In [ ]:
# [250228 cell 337]
def cohens_d(x, y):
    pooled_std = np.sqrt(((len(x)-1) * np.var(x, ddof=1) 
                          + (len(y)-1) * np.var(y, ddof=1)) / 
                             (len(x) + len(y) - 2))
    return (np.mean(x) - np.mean(y)) / pooled_std


# Cohen's d provides a standardized measure of the difference between two means. 
# A higher value indicates a larger difference between the groups relative to the variability within the groups. 
# This measure is useful for understanding the practical significance of the difference, beyond just statistical significance.

##############################################
# params

test_cond = 'HI'
ctrl_cond = 'LO'
condition_col = "Condition"

##############################################
# Preparing the data matrix

adata_flux_for_volcano = adata_flux.copy()

sc.pp.log1p(adata_flux_for_volcano)

test_cells = adata_flux_for_volcano.obs.index[adata_flux_for_volcano.obs[condition_col] == test_cond]
ctrl_cells = adata_flux_for_volcano.obs.index[adata_flux_for_volcano.obs[condition_col] == ctrl_cond]

print(len(test_cells))
print(len(ctrl_cells))

df = pd.DataFrame(adata_flux_for_volcano.X.T,
                  index = adata_flux_for_volcano.var.index,
                  columns=adata_flux_for_volcano.obs.index)

print(df.head())

test_df = df.loc[:,test_cells]
ctrl_df = df.loc[:,ctrl_cells]

for flux_id in df_l_test2.index:
    A, B = test_df.loc[flux_id].to_numpy().ravel(), ctrl_df.loc[flux_id].to_numpy().ravel()
    c_d = cohens_d(A, B)
    df_l_test2.loc[flux_id, ['cohens_d']] = c_d

df_l_test2.index.name = 'M_id'

In [ ]:
# [250228 cell 339]
df_l_test2.to_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/LR_test_matrix_for_volcano_plot_250203.csv")

Plotting step.

In [ ]:
# [250228 cell 357]
df_l_test2 = pd.read_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/LR_test_matrix_for_volcano_plot_250203.csv",index_col=0)
df_l_test2.head()

meta_new = pd.read_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/scFEA_analysis_240731/scFEA.M171.mouse.moduleinfo_AddAnno_2.csv",index_col=0)
meta_new = meta_new.rename(columns={"SM_anno":"SM_anno2"})
meta_new.head()

df_l_test2 = pd.merge(df_l_test2,meta_new[["SM_anno2"]],left_index=True,right_index=True,how="left")
df_l_test2
df_l_test2 = df_l_test2.reset_index()

df_l_test2.index = df_l_test2["M_id"]

data_1 = df_l_test2.loc[df_l_test2['SM_anno']=="Glycolysis_TCA_cycle"]
cohen_thr = 0.15
#label_data = data_1[(data_1['p_val']<0.05) & (abs(data_1['cohens_d'])>cohen_thr) & (data_1["SM_anno2"]=="TCA_cycle")]

label_data = data_1[(abs(data_1['cohens_d'])>cohen_thr) & (data_1["SM_anno2"]=="TCA_cycle")]


label_data["color_column"] = np.where(label_data["SM_anno2"]=="TCA_cycle","#4DBBD5","#F39B7F")#"#FF0000","#00FF00")
label_data

label_data_TCA = label_data[label_data["SM_anno2"]=="TCA_cycle"]
label_data_Glycolysis = label_data[label_data["SM_anno2"]=="Glycolysis"]

label_data1 = label_data[label_data["cohens_d"]<0]
label_data2 = label_data[label_data["cohens_d"]>0]

In [ ]:
# [250228 cell 361]
%%R -i df_l_test2 -i label_data_TCA -i label_data_Glycolysis -i label_data -i label_data1 -i label_data2

library(ggplot2)
library(dplyr)
library(ggrepel)

# Assuming df_l_test2 is your data frame in R
data <- df_l_test2

# Function to plot Super Module
plot_SM <- function(SM, cohen_thr = 0.15) {
  data$p_val_adj[data$p_val_adj < 1e-300] <- 1e-300
  data_1 <- data[data$SM_anno == SM, ]
  data_2 <- data[data$SM_anno != SM, ]
  
  p <- ggplot() +
    geom_point(data = data_2, aes(x = cohens_d, y = -log10(p_val)), color = "#CCCCCC") +
    geom_point(data = data_1, aes(x = cohens_d, y = -log10(p_val)), color = "#b22222") +
    labs(y = "-log10 (LR pvalue)", 
         title = paste(SM, "reactions in red"),
         x = "Cohen's d") +
    theme_classic() +
    theme(
    plot.title = element_text(size = 14, face = "bold", hjust = 0.5, margin = margin(b = 1, t = 0), color = "black"),
    legend.position = c(0.43, 1.04),  # Position legend in top right corner
    legend.justification = c(1, 1),   # Align legend to top right corner
    legend.box.just = "right",
    legend.margin = margin(6, 6, 6, 6),
    #legend.position = "right",
    legend.text = element_text(size = 12),  # Increase legend text size
    legend.title = element_blank(),  # 
    panel.grid = element_blank(),  # Remove grid
    axis.line = element_line(color = "black"),  # Add axis lines
    plot.background = element_rect(fill = "white", color = NA),  # White background
    panel.background = element_rect(fill = "white", color = NA),  # White panel
    axis.ticks = element_line(color = "black"),  # Add black ticks
    axis.ticks.length = unit(0.2, "cm"),  # Set tick length
    axis.text = element_text(size = 11),  # Increase axis text size
    axis.title = element_text(size = 12),  # Increase axis title size
    axis.text.x.top = element_blank(),  # Remove top x-axis labels
    axis.ticks.x.top = element_blank(),  # Remove top x-axis ticks
    axis.line.x.top = element_line(color = "black"),
    axis.text.y.right = element_blank(),  # Remove right y-axis labels
    axis.ticks.y.right = element_blank(),  # Remove right y-axis ticks
    axis.text.x = element_text(size = 13),
    axis.text.y = element_text(size = 13),
    axis.title.x = element_text(size = 13.5, margin = margin(t = 5)),  # Increase x-axis label size and add margin
    axis.title.y = element_text(size = 13.5, margin = margin(r = 5)),  # Increase y-axis label size and add margin
    )
  
  x_lim <- 1 #max(abs(max(data$cohens_d)) + 0.1, 0.6)
  p <- p + 
    geom_vline(xintercept = c(-cohen_thr, cohen_thr), linetype = "dashed", color = "grey") +
    geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "grey") +
    annotate("text", x = -0.82, y = -log10(0.05) + 0.15, label = "pvalue 0.05", size = 3.5,color="dimgrey")
  
  # Add arrows and labels
  y2=0.33
  y1=0.15
  p <- p +
    annotate("segment", x = 0, xend = 1, y = -y1, yend = -y1, 
             arrow = arrow(length = unit(0.2, "cm")), color = "#b22222", size = 1) +##4682B4
    annotate("text", x = 0.75, y = -y2, label = expression("Biotin"^italic(Hi)), parse = TRUE, size = 4.5) +
    annotate("segment", x = 0, xend = -1, y = -y1, yend = -y1, 
             arrow = arrow(length = unit(0.2, "cm")), color = "#4682B4", size = 1) +
    annotate("text", x = -0.75, y = -y2, label = expression("Biotin"^italic(Lo)), parse = TRUE, size = 4.5)
  
  # Add labels for significant points

  #label_data <- data_1[data_1$reaction %in% labeled_reactions, ]
  print(label_data)
  
  p <- p + geom_text_repel(
    data = label_data,
    aes(x = cohens_d, y = -log10(p_val),color = SM_anno2,label = paste(M_id, ":", M_name)),
    size = 4,  # Increase text size
    box.padding = 0.1,
    point.padding = 0.1,
    #nudge_x = 1, #make texts align to x coord
    direction = "y",
    hjust = 0.1,
    force = 18,  # Increase force to spread labels more
    segment.size = 0.4,
    segment.color = "grey50",
    min.segment.length = 0.1,
    max.overlaps = w,  # Limit overlaps
    show.legend = TRUE
  )+scale_color_identity()+scale_color_manual(values = c("TCA_cycle" = "#4DBBD5", "Glycolysis" = "#F39B7F"))
  
  return(p)
}

# Create the plot
volcano_plot <- plot_SM("Glycolysis_TCA_cycle")

volcano_plot <- volcano_plot +
  coord_cartesian(
    xlim = c(-1, 1),  # Set x-axis limits
    ylim = c(-0.4, 2.5)   # Set y-axis limits
  ) +
  scale_x_continuous(breaks = seq(-1, 1, by = 0.5),sec.axis = dup_axis(name = NULL) ) +  # Set x-axis breaks
  scale_y_continuous(breaks = seq(0, 4, by = 1),minor_breaks = seq(0, 4, by = 1),sec.axis = dup_axis(name = NULL))+    # Set y-axis breaks
  theme(
  legend.background = element_rect(fill = "transparent", color = NA),
  legend.key = element_rect(fill = "transparent", color = NA),
  legend.position = c(0.27, 1.05),
  legend.text = element_text(size = 9,color="dimgrey",margin = margin(l = -8, unit = "pt")),
  )


# Display the plot
print(volcano_plot)

# Save the plot if needed
ggsave("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_4I_TCA_volcano_plot_250205.pdf", volcano_plot, width = 4.2, height = 4.2, dpi = 300)


## Figure 5A, MHC Class II core enrichment gene score


In [ ]:
# [250228 cell 266]
adata_filter_HI_HC = sc.read_h5ad("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/adata_filter_HI_HC_250202.h5ad")

In [ ]:
# [250228 cell 281]
l_6genes_mouse = ['Cd74',
 'H2-Aa',
 'H2-Eb1',
 'H2-Ab1',
 'Ctss',
 'Tubb1']


In [ ]:
# [250228 cell 288]
sc.tl.score_genes(adata_filter_HI_HC,gene_list=l_6genes_mouse,score_name='MHC_Class2_Core_Enrichment',use_raw=False)

In [ ]:
# [250228 cell 289]
MHC_Class2_Core_Enrichment_score = adata_filter_HI_HC.obs[["MHC_Class2_Core_Enrichment"]]
MHC_Class2_Core_Enrichment_score.head()

In [ ]:
# [250228 cell 291]
df_tmp = adata_filter_HI_HC.obs[["BC","MHC_Class2_Core_Enrichment"]]
df_tmp.head()

In [ ]:
# [250228 cell 292]
df_tmp["Class"] = "MHC_Class2_Core_Enrichment"

In [ ]:
# [250228 cell 297]
df_tmp.to_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_4G_for_plot_df_tmp_250302.csv")

In [ ]:
# [250331 cell 389]
df_tmp = pd.read_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_4G_for_plot_df_tmp_250302.csv",index_col=0)
df_tmp

In [ ]:
# [250331 cell 390]
%%R -i df_tmp

df_1_2_v1 = df_tmp

library(ggplot2)
library(ggpubr)
library(dplyr)
library(tidyr)

# Assuming your data is in a dataframe called df_1_2_v1
# If not, you'll need to load your data first

HI_Control_color = "#015493"
HI_Infected_color = "#941914"
LO_Control_color = "#42257e"
LO_Infected_color = "#009193" 



# Define colors
d_colors <- c(
  "HI_Control" = HI_Control_color,
  "HI_Infected" = HI_Infected_color
)

d_colors <- c(
  "HI_Control" = HI_Control_color,
  "HI_Infected" = HI_Infected_color
)


# Your existing ggplot code
p <- ggplot(df_1_2_v1, aes(x = `BC`, y = `MHC_Class2_Core_Enrichment`, fill = `BC`,color=`BC`)) +
  geom_jitter(width = 0.1, size = 0.8, alpha = 0.6, color = "black") +
  geom_violin(scale = "width", size=0.8, alpha = 0.2, width = 0.7, adjust = 1, trim = FALSE) +
  geom_boxplot(width = 0.15,color="white",alpha = 0.6,outlier.shape=NA) +#, color="white"
  #geom_hline(data = median_values, aes(yintercept = median_score),linetype = "dashed", color = "black", alpha = 0.7) +
  scale_fill_manual(values = d_colors,guide = "none") +  # Fill colors with transparency
  scale_color_manual(values = d_colors, guide = "none") +
  scale_x_discrete(limits = c("HI_Control", "HI_Infected"),
                   labels = c(
                     expression(atop("Control", Biotin^Hi)),#Biotin^italic(Hi)
                     expression(atop("Infected", Biotin^Hi)))
                   ) +
# Create the plot
# p <- ggplot(df_1_2_v1, aes(x = `BC`, y = `MHC_Class2_Core_Enrichment`, fill = `BC`)) +
  # geom_violin(scale = "width", alpha = 0.9, width = 0.65,adjust = 1,trim=FALSE) +
  # #geom_dotplot(binaxis='y', stackdir='center', dotsize=0.6)+
  # #coord_flip() +
  # #geom_point() +
  # geom_jitter(width = 0.1, size = 0.8, alpha = 0.8, color = "black") +
  # geom_boxplot(width = 0.1, color = 'white', alpha = 0.2) +
  #scale_fill_manual(values = d_colors) +
  #scale_x_discrete(limits = c("HI_Control", "HI_Infected"),
  #                 labels = c("Control\nBiotin^Hi", "Infected\nBiotin^Hi")) +
#scale_y_continuous(breaks=c(-3,-2,-1,0,1,2,3,4,5),labels=c("-3","-2","-1","0","1","2","3","4","5"))+
theme_classic()+
  theme(
    legend.position = "none",
    axis.title.x = element_blank(),
    axis.text.x = element_text(size = 13, angle = 0, hjust = 0.5),
    #axis.text.x = element_blank(),
    axis.ticks.x = element_blank(),
    axis.text.y = element_text(size = 12),
    axis.title.y = element_text(size = 14),
    #panel.grid.major = element_blank(),
    #panel.grid.minor = element_blank()
  ) +
  labs(y = paste("MHC Class",expression(II),"Core Enrichment","\n","Gene Score")) +
  coord_cartesian(ylim = c(-2.9, 5.2), xlim = c(0.95, 2.05)) 
  #geom_hline(yintercept = median(df_1_2_v1$`MHC_Class2_Core_Enrichment`), linetype = "dashed", color = "black", alpha = 0.7)

# Add statistical annotations
#p <- p + stat_compare_means(
#  comparisons = list(c("HI_Infected", "HI_Control")),
#  method = "wilcox.test",
#  label = "p = {p}",#"p.signif",
#  label.y = c(3.1, 3.2),size=4.5
#)
#

p<- p + geom_pwc(
    aes(group = BC),tip.length = 0, method = "wilcox_test", label = "p",bracket.nudge.y = 0.28,step.increase=0.132

  )#+
  #labs(y = "log-normalized expression")


#p <- p+annotate("text", x = 2.3, y = 1, 
#             label =  paste("Cd74","H2-Aa","H2-Eb1","H2-Ab1","Ctss","Tubb1",
#"Ctsk",
#"H2-DMa",
#"H2-DMb1",
#"H2-0a",sep="\n"),
#             hjust = 0, vjust = 0.5, size = 4, color = "red")

# Print the plot
print(p)

# Save the plot
ggsave("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_4G_violin_plot_MHC_Class2_250429.pdf", p, width = 3.5, height = 3.5, dpi = 500)

## Supplemental Figure 5A, HSC score and Repopulation score
The HSC score comes from the hscScore model, which runs outside the notebook.
Both scores are computed on all 173 cells, and the plot shows the three groups.
Writes `FigS3A_2scores_250429.pdf`.

In [ ]:
# [250228 cell 138]
def cal_repop():
    RepopSig_genes = np.genfromtxt(RepopSig_genes_txt,dtype="str")

    adata_qc = sc.read_h5ad("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/adata_qc_173_corrected_Biotin_250121.h5ad")
    print(adata_qc.shape)
    print(adata_qc.X.max())
    adata_qc.var_names = adata_qc.var_names.astype(str)
    adata_qc.var_names_make_unique()
    smqpp.normalise_data(adata_qc)
    print(adata_qc.shape)
    print(adata_qc.X.max())
    
    missingGenes = np.setdiff1d(RepopSig_genes,adata_qc.var_names)
    if missingGenes.size != 0:
        print(f"#Missed Genes:{missingGenes}")
        
    sc.tl.score_genes(adata_qc,RepopSig_genes,score_name='Repop',use_raw=False)
    RepopSig_score =adata_qc.obs[["Repop"]]
    print(RepopSig_score.head())
    return RepopSig_score

In [ ]:
# [250228 cell 139]
RepopSig_score = cal_repop()
RepopSig_score.to_csv('/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/SLX21561_RepopSig_score_250125.csv')

In [ ]:
# [250228 cell 143]
def total_count_normalise(count_matrix):
    """Normalise count matrix for input into hscScore model.
    Performs read depth normalisation normalising each cell so that normalised 
    counts sum to the same value.
    
    Parameters
    ----------
    count_matrix : pandas dataframe
        Gene count matrix of dimension cells x genes with column names as genes
        and index as cell names
    
    Returns
    -------
    **norm_matrix** : pandas dataframe
        Normalised count matrix of dimension cells x genes
    """
    
    # Set the value normalised counts will sum to for each cell
    wilson_molo_genes_median_counts = 18704.5
    
    # Scale rows
    count_matrix_expression = np.array(count_matrix, dtype='float')
    counts_per_cell = np.sum(count_matrix_expression, axis=1)
    counts_per_cell += (counts_per_cell == 0)
    counts_per_cell /= wilson_molo_genes_median_counts
    norm_matrix_expression =  count_matrix_expression/counts_per_cell[:, None]
    norm_matrix = pd.DataFrame(norm_matrix_expression, index=count_matrix.index,
                               columns=count_matrix.columns)
    # log + 1 transform the data
    norm_matrix = np.log(norm_matrix + 1)
    
    return norm_matrix


#########
def generate_cal_hscscore_csv(adata):#should be the raw count
    HSCscore_genes = pd.Series(np.genfromtxt('/home/ql321/ql321/files/model_molo_genes.txt', dtype='str'))
    print(HSCscore_genes.shape)
    missing = HSCscore_genes[~HSCscore_genes.isin(adata.var.index)]
    print(missing)

    tempX = np.concatenate((adata.X, np.zeros((adata.n_obs, len(missing)))),
               axis = 1)
    print(tempX.shape)

    temp = sc.AnnData(X = tempX,
              obs = adata.obs,
              var = pd.DataFrame(index = np.concatenate((adata.var.index.values, missing.values))))

    HSCscore_genes = np.genfromtxt('/home/ql321/ql321/files/model_molo_genes.txt', dtype='str')
    print(HSCscore_genes.shape)
    vwf_index = np.where(HSCscore_genes=="Vwf")[0]
    print(vwf_index)
    X = temp[:,HSCscore_genes].X.copy()
    X = pd.DataFrame(X)
    #X[vwf_index] = 0
    print(X.loc[:,~X.any()])
    
    X = total_count_normalise(X)
    
    print(X.shape)
    X.to_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/for_hscScore_cal_250228.csv",index=True,header=True)
    return X
    
X = generate_cal_hscscore_csv(adata)

In [ ]:
# [250228 cell 145]
tmp_hscScore = pd.read_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/hscscore_result_250228.csv",header=0,names=["Sequencing_identifier","HSC_score"])
tmp_hscScore

In [ ]:
# [250228 cell 147]
adata_umap.obs['HSC_score'] = tmp_hscScore.HSC_score.values#calculate_HSCscore(temp)

In [ ]:
# [250228 cell 151]
merged_df = pd.concat([p53_score, AG_score, RepopSig_score], axis=1, join='inner')
merged_df

In [ ]:
# [250228 cell 155]
adata_umap.obs = adata_umap.obs.rename(columns={"Biotin Condition":"BC","HSC_score":"hscScore"})

In [ ]:
# [250228 cell 156]
df = sc.get.obs_df(adata_umap,["BC","Biotin","Condition",
                                 "hscScore","p53S",
                                 "AG_score","Repop"])
df

In [ ]:
# [250228 cell 158]
df_tmp = df[["BC","hscScore"]].rename(columns={"hscScore":"score"})
df_tmp["Class"] = "HSC Score"
df_hscscore = df_tmp.copy()
df_hscscore.head()

In [ ]:
# [250228 cell 161]
df_tmp = df[["BC","Repop"]].rename(columns={"Repop":"score"})
df_tmp["Class"] = "Repopulation Score"
df_Repop = df_tmp.copy()
df_Repop.head()

In [ ]:
# [250228 cell 162]
dfs = pd.concat([df_hscscore,df_p53S,df_aging,df_Repop])
dfs.shape

In [ ]:
# [250228 cell 163]
dfs.to_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/FigS1_plot_250228.csv")

In [ ]:
# [250331 cell 69]
dfs = pd.read_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/FigS1_plot_250228.csv",index_col=0)
dfs

In [ ]:
# [250331 cell 75]
dfs = dfs[dfs["Class"].isin(["HSC Score", "Repopulation Score"])].copy()

In [ ]:
# [250331 cell 76]
%%R -i dfs
## ------------------------------------------------------------------ ##
## 1 · libraries                                                      ##
## ------------------------------------------------------------------ ##
library(ggplot2)
library(ggpubr)          # geom_pwc()
library(dplyr)
library(ggh4x)
## ------------------------------------------------------------------ ##
## 2 · colours                                                        ##
## ------------------------------------------------------------------ ##
cols <- c(
  HI_Control  = "#015493",
  HI_Infected = "#941914",
  LO_Infected = "#009193"
)

## ------------------------------------------------------------------ ##
## 3 · base plot                                                      ##
## ------------------------------------------------------------------ ##
dfs$Class <- factor(dfs$Class, levels = c("HSC Score", "Repopulation Score"))
dfs$BC <- factor(dfs$BC, levels = c("HI_Control", "HI_Infected", "LO_Infected"))

p <- ggplot(dfs, aes(BC, score, fill = BC, colour = BC)) +
  geom_jitter(width = 0.1, size = .8, alpha = .6, colour = "black") +
  geom_violin(scale = "width", width = .7, trim = FALSE,
              alpha = .2, size = .8) +
  geom_boxplot(width = .15, alpha = .6, outlier.shape = NA,
               colour = "white") +
  scale_fill_manual(values = cols, guide  = "none") +
  scale_colour_manual(values = cols, guide = "none") +
  scale_x_discrete(
    limits = c("HI_Control", "HI_Infected", "LO_Infected"),
    labels = c(
      expression(atop("Control",  Biotin^Hi)),
      expression(atop("Infected", Biotin^Hi)),
      expression(atop("Infected", Biotin^Lo))
    )
  ) +
  ## per-panel padding (bottom 5 %, top 30 %)
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.11))) +
  ## Wilcoxon tests per facet
  geom_pwc(aes(group = BC),
           method          = "wilcox_test",
           label           = "p",
           tip.length      = 0,
           bracket.nudge.y = 0.24,
           step.increase   = 0.132) +
  labs(x = NULL, y = NULL) +                         # y-title via strip
  theme_classic(base_size = 13) +
  theme(
    axis.text.x  = element_text(size = 13, vjust = 0.5),
    axis.ticks.x = element_line(size = 0.4),

    strip.placement    = "outside",
    strip.background   = element_rect(fill = "white", colour = NA),
    strip.text.y.left  = element_text(angle = 90, size = 13),
    plot.margin        = margin(t = 8, r = 5, b = 5, l = 5)
  ) +
  facet_wrap2(~ Class, nrow = 1, strip.position = "left",scales = "free_y",axes = "x",remove_labels  = "x")

print(p)
## ------------------------------------------------------------------ ##
## 4 · export                                                         ##
## ------------------------------------------------------------------ ##
ggsave("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/FigS3A_2scores_250429.pdf", p, width = 6, height = 3, dpi = 500)


## Figure 4I right, OXPHOS gene score violin plot

Uses the same 165 cell object and the same normalisation as panels B to H.
The gene set holds 148 genes, and 128 of them are present in the 165 cell object.
`sc.tl.score_genes` draws 1046 control genes.
Writes `df_OXPHOS_Gene_Score_251214.csv` and `Fig_S_violin_plot_OXPHOS_251214.pdf`.

In [ ]:
# [OXPHOS cell 5]
oxphos = "Atp6-ps,Gadd45gip1,Mir451b,Ndufb1,Chchd10,Ndufb11,Ppif,Uqcc3,Shmt2,Actn3,Ak4,Abcd1,Apoc3,Rhoa,Atp5f1a,Atp5f1b,Atp5f1c,Atp5pb,Atp5pf,Atp5me,Atp7a,Bdnf,Bid,Cdk1,Coq7,Cox4i1,Cox5a,Cox5b,Cox6a1,Cox6a2,Cox7a1,Cox7a2,Cox7c,Cox8a,Cox8b,Cycs,Cyct,Dld,Chchd2,Fxn,Nipsnap2,Msh2,mt-Atp6,mt-Atp8,mt-Co1,mt-Co2,mt-Co3,mt-Cytb,mt-Nd1,mt-Nd2,mt-Nd3,mt-Nd4,mt-Nd4l,mt-Nd5,mt-Nd6,Myc,Myog,Ndufa2,Ndufs4,Ndufv1,Cox7a2l,Snca,Pde2a,Antkmt,Afg1l,Tnf,Uqcrq,Uqcrc1,Ndufs8,Ndufs2,Ndufs1,Ndufb6,Cimap2,Ccnb1,Macroh2a1,Vcp,Dguok,Atp5po,Ndufs6,Chchd2-ps,Ccnb1-ps,Ndufa1,Nupr1,Mtch2,Park7,Atp5mf,Mlxipl,Ndufs5,Atp5f1d,Ndufb5,Sdhc,Sdhaf2,Ndufa3,Ndufa9,Dnajc30,Cox7b,Dnajc15,Uqcr10,Ndufb9,Ndufc1,Iscu,Ndufa12,Ndufa7,Cyc1,Ndufb3,Uqcrh,Stoml2,Uqcr11,Uqcrfs1,Tafazzin,Ndufb7,Sdhd,Sdha,Slc25a23,Uqcrc2,Atp5f1e,Ndufa6,Ndufa13,Ndufb8,Uqcc2,Ndufa10,Uqcrb,Sdhb,Coa6,Coq9,Atpsckmt,Ndufb4,Ndufc2,Ndufb2,Ndufa5,Ndufb10,Ndufs3,Ndufa8,Tefm,Pink1,Ndufaf1,Ndufa11,Ndufab1,Slc25a33,Atp5pd,Mir451a,Tmem135,Ndufv2,1700066M21Rik,Ndufs7,Cox8c,Ndufv3,Cox4i2"
oxphos_gene_list = oxphos.split(",")
oxphos_gene_list

In [ ]:
# [OXPHOS cell 47]
adata_qc_selected_cells = sc.read_h5ad("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/adata_qc_173_corrected_Biotin_250121.h5ad")
adata_qc_165 = adata_qc_selected_cells[adata_qc_selected_cells.obs["Biotin Condition"]!="LO_Control"].copy()
sc.pp.filter_genes(adata_qc_165,min_cells=1)
adata_qc_165.obs["BC"] = adata_qc_165.obs["Biotin Condition"].copy()

adata_qc_165.var_names = adata_qc_165.var_names.astype(str)
adata_qc_165.var_names_make_unique()

print(adata_qc_165.X.max())
adata_qc_165_bf_norm = adata_qc_165.copy()
smqpp.normalise_data(adata_qc_165)

adata_qc_165.X.max()

In [ ]:
# [OXPHOS cell 51]
l_genes = list(adata_qc_165.var_names.intersection(oxphos_gene_list))

In [ ]:
# [OXPHOS cell 52]
len(l_genes)

In [ ]:
# [OXPHOS cell 54]
sc.tl.score_genes(adata_qc_165,gene_list=l_genes,score_name='OXPHOS Gene Score',use_raw=False)

In [ ]:
# [OXPHOS cell 55]
sc.pl.violin(adata_qc_165,keys="OXPHOS Gene Score",groupby="BC",
             palette={'HI_Control' : "#015493",
'HI_Infected': "#941914",
'LO_Infected': "#009193" },rotation=90)

In [ ]:
# [OXPHOS cell 56]
df_tmp = adata_qc_165.obs[["BC","OXPHOS Gene Score"]]
df_tmp.head()

In [ ]:
# [OXPHOS cell 58]
df_tmp.to_csv("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/df_OXPHOS_Gene_Score_251214.csv")

In [ ]:
# [OXPHOS cell 59]
%%R -i df_tmp

df_1_2_v1 = df_tmp

library(ggplot2)
library(ggpubr)
library(dplyr)
library(tidyr)

# Assuming your data is in a dataframe called df_1_2_v1
# If not, you'll need to load your data first

HI_Control_color = "#015493"
HI_Infected_color = "#941914"
LO_Control_color = "#42257e"
LO_Infected_color = "#009193" 



# Define colors
d_colors <- c(
  "HI_Control" = HI_Control_color,
  "HI_Infected" = HI_Infected_color,
  "LO_Infected" = LO_Infected_color
)


# Your existing ggplot code
p <- ggplot(df_1_2_v1, aes(x = `BC`, y = `OXPHOS Gene Score`, fill = `BC`,color=`BC`)) +
  geom_jitter(width = 0.1, size = 0.8, alpha = 0.6, color = "black") +
  geom_violin(scale = "width", size=0.8, alpha = 0.2, width = 0.7, adjust = 1, trim = FALSE) +
  geom_boxplot(width = 0.15,color="white",alpha = 0.6,outlier.shape=NA) +#, color="white"
  scale_fill_manual(values = d_colors,guide = "none") +  # Fill colors with transparency
  scale_color_manual(values = d_colors, guide = "none") +
  scale_x_discrete(limits = c("HI_Control", "HI_Infected", "LO_Infected"),
                   labels = c(
                     expression(atop("Control", Biotin^Hi)),#Biotin^italic(Hi)
                     expression(atop("Infected", Biotin^Hi)),
                     expression(atop("Infected", Biotin^Lo))
                   )) +
theme_classic()+
  theme(
    legend.position = "none",
    axis.title.x = element_blank(),
    axis.text.x = element_text(size = 13, angle = 0, hjust = 0.5),
    #axis.ticks.x = element_blank(),
    axis.text.y = element_text(size = 12),
    axis.title.y = element_text(size = 14)) +
  labs(y = paste("OXPHOS Gene Score")) 

y_max <- max(df_1_2_v1$`OXPHOS Gene Score`)

p1 <- p + geom_pwc(
    aes(group = `BC`),tip.length = 0, method = "wilcox_test", label = "p",bracket.nudge.y = 0.29,step.increase=0.12

  )+
  coord_cartesian(ylim = c(NA, y_max + 0.6), clip = "off") +  # extra head-room
  theme(plot.margin = margin(t = 8, r = 5, b = 5, l = 5))

print(p1)#+

#ggsave("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_S_violin_plot_OXPHOS_250429.pdf", p1, width = 3.5, height = 3.5, dpi = 500)
ggsave("/rds/project/rds-SDzz0CATGms/users/ql321/work/work_SLX21561_SLX21562/analysis_231125_new_metadata_table/re_run_240628/Fig_S_violin_plot_OXPHOS_251214.pdf", p1, width = 3.5, height = 3.5, dpi = 500)